# SelvaSonic — Evaluación Rigurosa del Modelo Attention (Semana 5.1)

## Objetivo

Replicar el análisis riguroso del notebook 05 sobre el modelo de attention. Mismas métricas, mismo test set, misma metodología — para que la comparación con el baseline sea **honesta y reproducible**.

## Métricas que vamos a calcular

Idénticas a las del notebook 05:
- Curvas Precision-Recall por clase + Average Precision (AP)
- Curvas ROC por clase + AUC-ROC
- Tabla completa de métricas
- Comparación AP vs AUC para detectar artefactos del desbalance

## Por qué hacer este notebook separado en vez de modificar el 05

Mantener un notebook por modelo y un notebook de comparación separado tiene varias ventajas:
1. **Reproducibilidad**: cada notebook se ejecuta independientemente y deja artefactos en la carpeta del run correspondiente.
2. **Auditabilidad**: si el profesor quiere ver "solo" las métricas del attention, abre este notebook.
3. **Comparación a posteriori**: el notebook 09 leerá los CSVs que ambos generan y los superpondrá.

## Sección 1 — Setup

Diferencias respecto al notebook 05:
- Cargamos `SelvaSonicCNNAttention` en lugar de `SelvaSonicCNN`.
- Apuntamos a la carpeta del run `attention_S4_v1_20260601_0334`.
- Mismo `random_state=42` → mismo test set → comparación honesta.

In [ ]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / 'src').exists():
    sys.path.insert(0, str(PROJECT_ROOT))
elif (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    roc_curve, auc, roc_auc_score
)
from sklearn.preprocessing import label_binarize

from src.dataset import create_dataloaders
from src.model import SelvaSonicCNNAttention  # <-- modelo de attention

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {device}')

COLOR_PRIMARY = '#6C5CE7'
COLOR_ACCENT = '#00CEC9'
COLOR_WARN = '#FD79A8'
COLOR_DARK = '#2D3436'

# Apuntar a la carpeta del run del attention
RUN_DIR = PROJECT_ROOT / 'results' / 'runs' / 'attention_S4_v1_20260601_0334'
BEST_CKPT = RUN_DIR / 'best.pth'
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'

assert BEST_CKPT.exists(), f'Falta {BEST_CKPT}'
assert RAW_DATA_DIR.exists(), f'Falta {RAW_DATA_DIR}'
print(f'\nUsando run: {RUN_DIR.name}')

In [ ]:
# Mismo split que el entrenamiento
_, _, test_loader, label_map = create_dataloaders(
    raw_data_dir=str(RAW_DATA_DIR),
    batch_size=32, num_workers=0,
    train_ratio=0.70, val_ratio=0.15, test_ratio=0.15,
    random_state=42, verbose=True,
)
NUM_CLASSES = len(label_map)
idx_to_name = {v: k for k, v in label_map.items()}
class_names = [idx_to_name[i] for i in range(NUM_CLASSES)]

# Cargar modelo attention
ckpt = torch.load(BEST_CKPT, map_location=device)
model = SelvaSonicCNNAttention(num_classes=NUM_CLASSES).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"\nModelo Attention cargado (epoca {ckpt['epoch']+1}, val_acc={ckpt['best_val_acc']:.4f})")
print(f"Parametros: {model.count_parameters():,}")

In [ ]:
all_true, all_probs = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        probs = F.softmax(model(x), dim=1)
        all_true.append(y.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

y_true = np.concatenate(all_true)
y_probs = np.concatenate(all_probs)
y_pred = y_probs.argmax(axis=1)
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

print(f'Total clips test: {len(y_true)}')
print(f'Accuracy global Attention: {(y_true == y_pred).mean():.4f}')
print(f'Accuracy global Baseline (ref): 0.6322')

## Sección 2 — Curvas Precision-Recall por clase

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor('#FAFAFA')
cmap = plt.cm.tab20
colors = [cmap(i / NUM_CLASSES) for i in range(NUM_CLASSES)]

ap_per_class = {}
for i in range(NUM_CLASSES):
    precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_probs[:, i])
    ap = average_precision_score(y_true_bin[:, i], y_probs[:, i])
    ap_per_class[class_names[i]] = ap
    ax.plot(recall, precision, lw=2, color=colors[i], label=f'{class_names[i][:18]} (AP={ap:.2f})')

ax.set_xlabel('Recall', fontsize=12, color=COLOR_DARK)
ax.set_ylabel('Precision', fontsize=12, color=COLOR_DARK)
ax.set_title('Curvas Precision-Recall — ATTENTION MODEL\n(one-vs-rest)', fontsize=13, color=COLOR_DARK)
ax.legend(loc='lower left', fontsize=8, framealpha=0.95)
ax.grid(alpha=0.3); ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.savefig(RUN_DIR / 'precision_recall_curves.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()

print('\nAP POR CLASE (Attention):\n')
for name, ap in sorted(ap_per_class.items(), key=lambda x: -x[1]):
    barra = '█' * int(ap * 30)
    print(f'  {name:<25} AP={ap:.3f}  {barra}')

ap_macro = np.mean(list(ap_per_class.values()))
ap_weighted = average_precision_score(y_true_bin, y_probs, average='weighted')
ap_micro = average_precision_score(y_true_bin, y_probs, average='micro')
print(f'\nAP macro: {ap_macro:.4f}  (Baseline: 0.6390 aprox)')
print(f'AP weighted: {ap_weighted:.4f}')
print(f'AP micro: {ap_micro:.4f}')

## Sección 3 — Curvas ROC y AUC

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor('#FAFAFA')

auc_per_class = {}
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)
    auc_per_class[class_names[i]] = roc_auc
    ax.plot(fpr, tpr, lw=2, color=colors[i], label=f'{class_names[i][:18]} (AUC={roc_auc:.2f})')

ax.plot([0, 1], [0, 1], '--', color=COLOR_DARK, alpha=0.5, label='Azar (AUC=0.5)')
ax.set_xlabel('FPR', fontsize=12, color=COLOR_DARK)
ax.set_ylabel('TPR (Recall)', fontsize=12, color=COLOR_DARK)
ax.set_title('Curvas ROC — ATTENTION MODEL', fontsize=13, color=COLOR_DARK)
ax.legend(loc='lower right', fontsize=8, framealpha=0.95)
ax.grid(alpha=0.3); ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.savefig(RUN_DIR / 'roc_curves.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()

auc_macro = roc_auc_score(y_true, y_probs, multi_class='ovr', average='macro')
auc_weighted = roc_auc_score(y_true, y_probs, multi_class='ovr', average='weighted')
print(f'\nAUC-ROC macro:    {auc_macro:.4f}')
print(f'AUC-ROC weighted: {auc_weighted:.4f}')

## Sección 4 — Tabla completa de métricas

In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

p = precision_score(y_true, y_pred, labels=range(NUM_CLASSES), average=None, zero_division=0)
r = recall_score(y_true, y_pred, labels=range(NUM_CLASSES), average=None, zero_division=0)
f1 = f1_score(y_true, y_pred, labels=range(NUM_CLASSES), average=None, zero_division=0)
support = np.array([(y_true == i).sum() for i in range(NUM_CLASSES)])

df = pd.DataFrame({
    'clase': class_names,
    'support': support,
    'precision': p,
    'recall': r,
    'f1': f1,
    'ap': [ap_per_class[n] for n in class_names],
    'auc_roc': [auc_per_class[n] for n in class_names],
}).sort_values('f1', ascending=False).reset_index(drop=True)

print('TABLA DE METRICAS — ATTENTION (ordenadas por F1):\n')
print(df.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

df.to_csv(RUN_DIR / 'metricas_completas.csv', index=False)
with open(RUN_DIR / 'metricas_completas.txt', 'w', encoding='utf-8') as f_out:
    f_out.write('METRICAS COMPLETAS — ATTENTION MODEL\n')
    f_out.write('=' * 70 + '\n')
    f_out.write(df.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
    f_out.write(f'\n\nAGREGADOS:\n')
    f_out.write(f'  AP-macro     : {ap_macro:.4f}\n')
    f_out.write(f'  AP-weighted  : {ap_weighted:.4f}\n')
    f_out.write(f'  AUC-ROC macro: {auc_macro:.4f}\n')
    f_out.write(f'  AUC-ROC wt   : {auc_weighted:.4f}\n')

print(f'\nGuardados: metricas_completas.csv y .txt en {RUN_DIR.name}/')

## Cierre

Este notebook deja en la carpeta del run del attention los mismos artefactos que el notebook 05 dejó para el baseline. El notebook 09 (comparación) leerá ambos `metricas_completas.csv` y generará las gráficas comparativas lado a lado.